# Advanced Python arithmetic operators: 18 problems with complete solutions

**Focus:** the arithmetic-operator material in the supplied lesson: `__add__`, `__sub__`, `__mul__`, `__rmul__`, dot and cross products, `__matmul__`, `__iadd__`, unary `__neg__`, `__abs__`, and nonnumeric operator meanings.

**Approach:** 18 progressively harder, fully solved problems; runnable solutions; worked examples; failure-path checks; geometry, algebra, numerics, and design exercises. The core `Vector` is created once and reused. Everything runs using the Python standard library (Python 3.10+ recommended). **Run All** from top to bottom. No network or external dataset required.

**Important conventions:** as in the supplied material, vector `*` means scalar scaling **or** dot product depending on the operand, and vector `@` is used for the **3D cross product**. The latter is a teaching convention, **not** the usual meaning of `@` in common numerical libraries, which use it for matrix multiplication. The later `Matrix` problem demonstrates conventional matrix `@`.

**Best-practice rules:** return `NotImplemented` for unsupported **binary operators**; raise explicit exceptions in constructors and ordinary methods; check dimensions before `zip`; decide whether `+=` mutates or rebinds and document it; keep arithmetic methods free of debug printing; avoid accidental `bool` components even though `bool` is a subclass of `int`; test valid inputs, invalid inputs, identity/aliasing, and algebraic laws. Floating-point assertions use tolerances where necessary.

**How to use:** Read each Challenge, try coding it yourself, then compare with its worked solution. The verification cells contain assertions and example output. Every verification cell should finish without an exception.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from itertools import chain
from math import acos, cos, fsum, hypot, isclose, pi, sin, sqrt
from numbers import Real
from random import Random
from timeit import timeit
from typing import Iterable


def expect_raises(exception_type, operation):
    """Assert that a zero-argument callable raises the expected exception."""
    try:
        operation()
    except exception_type as exc:
        return str(exc)
    except Exception as exc:
        raise AssertionError(f"Expected {exception_type.__name__}, got {type(exc).__name__}") from exc
    raise AssertionError(f"Expected {exception_type.__name__}, but no error occurred")


def vector_close(actual, expected, *, rel_tol=1e-12, abs_tol=1e-12):
    """Compare vectors componentwise without confusing equality and approximate equality."""
    assert len(actual) == len(expected)
    return all(isclose(x, y, rel_tol=rel_tol, abs_tol=abs_tol)
               for x, y in zip(actual.components, expected.components))

print("Standard-library setup complete.")

Standard-library setup complete.


---
## Problem 01 — Understand dispatch and `NotImplemented`

**Challenge.** Build instrumented operands to prove the order in which `a + b` tries `__add__` and `__radd__`. Next, test a proper subclass that overrides the reflected method; record the resulting dispatch order. Show what happens if both implementations decline the operation.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

A binary method returns the **singleton** `NotImplemented` to let Python attempt the other operand's reflected operation. This differs from **raising** `NotImplementedError`: the latter interrupts dispatch. For distinct operand types with a proper right-hand subclass overriding the reflected operation, Python may give that subclass's reflected operation priority. Inspect the trace rather than guessing.

In [2]:
events = []

class LeftOperand:
    def __add__(self, other):
        events.append("Left.__add__")
        return NotImplemented

class RightOperand:
    def __radd__(self, other):
        events.append("Right.__radd__")
        return "reflected success"

class BaseOperand:
    def __add__(self, other):
        events.append("Base.__add__")
        return "base success"

class DerivedOperand(BaseOperand):
    def __radd__(self, other):
        events.append("Derived.__radd__")
        return "subclass-first"

class RejectOperand:
    def __add__(self, other):
        return NotImplemented

    def __radd__(self, other):
        return NotImplemented

### Executable examples and verification

In [3]:
events.clear()
assert LeftOperand() + RightOperand() == "reflected success"
assert events == ["Left.__add__", "Right.__radd__"]
print("Ordinary dispatch:", events)

events.clear()
assert BaseOperand() + DerivedOperand() == "subclass-first"
assert events == ["Derived.__radd__"]
print("Subclass priority:", events)

message = expect_raises(TypeError, lambda: RejectOperand() + RejectOperand())
assert "unsupported operand" in message
print("Both rejected:", message)

Ordinary dispatch: ['Left.__add__', 'Right.__radd__']
Subclass priority: ['Derived.__radd__']
Both rejected: unsupported operand type(s) for +: 'RejectOperand' and 'RejectOperand'


**Key takeaway / extension.** Do not explicitly call `other.__radd__` in `__add__`; return `NotImplemented` and let Python manage precedence and fallback. The same logic applies to `-`, `*`, and `@` with their corresponding reflected methods.

---
## Problem 02 — Engineer an immutable, dimension-safe `Vector`

**Challenge.** Create a reusable vector type with nonempty, real-number components, an immutable tuple representation, readable `repr`, fixed dimensions, arithmetic and unary methods, scalar division, norm, equality and hashing. Reject booleans, unsupported operand types, and incompatible dimensions. Explain why immutable component storage alone is insufficient for full object immutability.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

A frozen, slotted dataclass prevents rebinding attributes and supplies value equality/hash for a tuple of components. Validate constructor arguments before assigning the tuple with `object.__setattr__`. **Dimension mismatches in binary operations return `NotImplemented`** here, preserving the lesson's eventual `TypeError` behavior; the constructor uses `ValueError` for an empty vector and `TypeError` for invalid types. This implementation allows any `numbers.Real` (other than `bool`); IEEE nonfinite inputs and floating overflow are not globally forbidden. `hypot(*components)` is usually more stable than `sqrt(sum(x*x ...))`. The `@` method deliberately implements only a 3D cross product. Scalar division rejects zero. `+=` is intentionally left to fall back to `+`, yielding a fresh immutable object.

In [4]:
@dataclass(frozen=True, slots=True, init=False)
class Vector:
    _components: tuple[Real, ...] = field(repr=False)

    def __init__(self, *components: Real):
        if not components:
            raise ValueError("A Vector needs at least one component")
        if any(isinstance(value, bool) or not isinstance(value, Real)
               for value in components):
            raise TypeError("All Vector components must be real, non-bool numbers")
        object.__setattr__(self, "_components", tuple(components))

    @property
    def components(self):
        return self._components

    def __repr__(self):
        return f"Vector{self._components!r}"

    def __len__(self):
        return len(self._components)

    def __iter__(self):
        return iter(self._components)

    def __getitem__(self, index):
        return self._components[index]

    def _compatible(self, other):
        return isinstance(other, Vector) and len(self) == len(other)

    def __add__(self, other):
        if not self._compatible(other):
            return NotImplemented
        return Vector(*(x + y for x, y in zip(self, other)))

    def __sub__(self, other):
        if not self._compatible(other):
            return NotImplemented
        return Vector(*(x - y for x, y in zip(self, other)))

    def __mul__(self, other):
        if isinstance(other, Real) and not isinstance(other, bool):
            return Vector(*(x * other for x in self))
        if self._compatible(other):
            return sum((x * y for x, y in zip(self, other)), 0)
        return NotImplemented

    def __rmul__(self, other):
        # Reflect only legitimate scalar * vector; never invent mixed-type addition.
        if isinstance(other, Real) and not isinstance(other, bool):
            return self * other
        return NotImplemented

    def __truediv__(self, other):
        if not isinstance(other, Real) or isinstance(other, bool):
            return NotImplemented
        if other == 0:
            raise ZeroDivisionError("Cannot divide a vector by zero")
        return Vector(*(x / other for x in self))

    def __neg__(self):
        return Vector(*(-x for x in self))

    def __pos__(self):
        return self  # Safe because this class is immutable.

    def __abs__(self):
        return hypot(*self._components)

    def __matmul__(self, other):
        if not self._compatible(other) or len(self) != 3:
            return NotImplemented
        ax, ay, az = self
        bx, by, bz = other
        return Vector(ay * bz - az * by,
                      az * bx - ax * bz,
                      ax * by - ay * bx)

### Executable examples and verification

In [5]:
v = Vector(3, 4)
assert repr(v) == "Vector(3, 4)"
assert len(v) == 2 and v[0] == 3 and tuple(v) == (3, 4)
assert v.components == (3, 4)
assert abs(v) == 5.0
assert Vector(3, 4) == v
assert hash(Vector(3, 4)) == hash(v)
assert +v is v and -v == Vector(-3, -4)
assert expect_raises(ValueError, lambda: Vector())
assert expect_raises(TypeError, lambda: Vector(True, 2))
assert expect_raises(TypeError, lambda: Vector(1, "2"))
assert expect_raises(TypeError, lambda: v + Vector(1, 2, 3))
assert expect_raises(TypeError, lambda: v + 7)
assert expect_raises(ZeroDivisionError, lambda: v / 0)
assert expect_raises(Exception, lambda: setattr(v, "_components", (99,)))
print("Immutable Vector ready:", v, "norm:", abs(v))

Immutable Vector ready: Vector(3, 4) norm: 5.0


**Key takeaway / extension.** `Vector.components` returns a tuple, not an exposed mutable list. The dataclass-generated equality/hash uses value semantics; in-place behavior is examined separately in Problem 9. The class is for real numbers, not a generic symbolic or complex-number vector.

---
## Problem 03 — Prove addition/subtraction closure and error safety

**Challenge.** Using the class above, compute `(a + b) - b`, verify subtraction order matters, and exercise every shape/type failure. Show that failure does not modify either operand. Explain why using `zip` without a dimension check would silently discard data.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

The two valid operations are coordinatewise: `(a+b)[i] = a[i]+b[i]` and `(a-b)[i] = a[i]-b[i]`. They are defined only at matching dimensions, and each creates a fresh `Vector`. `zip` stops at the shorter iterable; therefore the compatibility check must precede it. Valid types but mismatched lengths still result in `TypeError` after `NotImplemented` fallback, consistent with the provided lesson's convention.

In [6]:
def displacement(start: Vector, end: Vector) -> Vector:
    """Compute end - start with the Vector type's normal dispatch."""
    if not isinstance(start, Vector) or not isinstance(end, Vector):
        raise TypeError("start and end must be Vector instances")
    if len(start) != len(end):
        raise ValueError("displacement requires equal dimensions")
    return end - start

### Executable examples and verification

In [7]:
a = Vector(2, -3, 5)
b = Vector(-7, 10, 0)
assert a + b == Vector(-5, 7, 5)
assert (a + b) - b == a
assert a - b == Vector(9, -13, 5)
assert a - b != b - a
assert displacement(a, b) == b - a
assert a.components == (2, -3, 5) and b.components == (-7, 10, 0)
assert expect_raises(TypeError, lambda: a - Vector(1, 2))
assert expect_raises(TypeError, lambda: a - "bad")
assert expect_raises(ValueError, lambda: displacement(a, Vector(1)))
print("a+b:", a + b, "a-b:", a - b)
print("zip truncation (why to validate first):", list(zip((1, 2, 3), (4, 5))))

a+b: Vector(-5, 7, 5) a-b: Vector(9, -13, 5)
zip truncation (why to validate first): [(1, 4), (2, 5)]


**Key takeaway / extension.** Binary overloads and named domain methods can intentionally have different exception contracts: this `displacement` API explicitly raises `ValueError` for invalid dimensions; the operators use `NotImplemented` to preserve dispatch.

---
## Problem 04 — Resolve overloaded multiplication without ambiguity

**Challenge.** Work through `v * 3`, `3 * v`, `v * w`, unsupported multiplication, and the dot-product linearity identity `(u + v) * w = u*w + v*w`. Explain why `v * w` returns a scalar but `v * 3` returns a vector, and why both are not interchangeable in arbitrarily parenthesized expressions.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Dispatch first checks for a valid real scalar, next a compatible vector, otherwise declines. The result type is **operand-dependent**, so document it clearly. For compatible vectors `u*v = Σ uᵢvᵢ`; for scalars `c*u` scales every component. A reversed scalar multiplication routes to `__rmul__`, not an accidental second call to `__mul__` on the scalar. `int` arithmetic remains exact for typical integer examples.

In [8]:
def affine_combination(a: Real, u: Vector, b: Real, v: Vector) -> Vector:
    """Compute a*u + b*v with an explicit dimension and coefficient contract."""
    if any(isinstance(c, bool) or not isinstance(c, Real) for c in (a, b)):
        raise TypeError("Coefficients must be real non-bool numbers")
    if len(u) != len(v):
        raise ValueError("Vectors must have the same dimension")
    return a * u + b * v

### Executable examples and verification

In [9]:
u, v, w = Vector(1, 2, 3), Vector(4, -1, 2), Vector(2, 0, -1)
assert u * 3 == Vector(3, 6, 9)
assert 3 * u == Vector(3, 6, 9)
assert u * v == 8  # 4 - 2 + 6
assert (u + v) * w == u * w + v * w
assert affine_combination(2, u, -1, v) == 2 * u - v
assert expect_raises(TypeError, lambda: u * "3")
assert expect_raises(TypeError, lambda: "3" * u)
assert expect_raises(TypeError, lambda: u * Vector(1, 2))
assert expect_raises(TypeError, lambda: True * u)
print("scaled:", 3 * u, "dot:", u * v, "affine:", affine_combination(2, u, -1, v))

scaled: Vector(3, 6, 9) dot: 8 affine: Vector(-2, 5, 4)


**Key takeaway / extension.** Alternative professional API: use a named `dot(u, v)` or `u @ v` for dot products to avoid overloading `*` with two result types. Here we intentionally retain the source lesson's contract.

---
## Problem 05 — Implement and audit the 3D cross-product convention

**Challenge.** Show that `a @ b` implements the right-handed cross product, verify anti-commutativity and perpendicularity, and reject 2D, 4D, and non-vector arguments. Derive an oriented area from the cross-product norm.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

For `a=(a₁,a₂,a₃)` and `b=(b₁,b₂,b₃)`, the result is `(a₂b₃−a₃b₂, a₃b₁−a₁b₃, a₁b₂−a₂b₁)`. It is perpendicular to both inputs (up to rounding), and its magnitude equals the area of the parallelogram. The 3D-only check is essential: the source lesson demonstrates the `__matmul__` hook but does not complete its calculation. We complete it here, making the unconventional use explicit.

In [10]:
def parallelogram_area(a: Vector, b: Vector) -> float:
    """Area in 3D; incompatible dimensions remain operator TypeErrors."""
    return abs(a @ b)


def triangle_area(a: Vector, b: Vector) -> float:
    return parallelogram_area(a, b) / 2

### Executable examples and verification

In [11]:
ex, ey, ez = Vector(1, 0, 0), Vector(0, 1, 0), Vector(0, 0, 1)
assert ex @ ey == ez
assert ey @ ex == -ez
assert ex @ ex == Vector(0, 0, 0)
assert (ex @ ey) * ex == 0 and (ex @ ey) * ey == 0
assert parallelogram_area(Vector(3, 0, 0), Vector(0, 4, 0)) == 12
assert triangle_area(Vector(3, 0, 0), Vector(0, 4, 0)) == 6
for bad in (Vector(1, 2), Vector(1, 2, 3, 4), "wrong"):
    assert expect_raises(TypeError, lambda bad=bad: ex @ bad)
print("e_x @ e_y:", ex @ ey, "triangle area:", triangle_area(Vector(3, 0, 0), Vector(0, 4, 0)))

e_x @ e_y: Vector(0, 0, 1) triangle area: 6.0


**Key takeaway / extension.** Library interoperability warning: NumPy's `@` means matrix multiplication, and its cross product is normally a named function. Domain-specific operators should be documented prominently.

---
## Problem 06 — Derive projection, rejection, and angle

**Challenge.** Write `project(u, onto)`, `reject(u, onto)`, and `angle_between(u, v)` using the overloaded operators. Guard against zero vectors and mismatched dimensions. Prove that `u = projection + rejection`, and check projection/rejection orthogonality.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

The projection formula is `proj_v(u) = ((u·v)/(v·v))v`. Rejection is `u−proj_v(u)`; its dot product with `v` is approximately zero. The angle formula is `acos((u·v)/(|u||v|))`; floating error may push the cosine outside `[-1,1]`, so clamp it before `acos`. Zero-length inputs have no direction and must be rejected. These demonstrations use finite, moderate values; avoiding dot-product overflow for extreme magnitudes needs a stronger numerical design.

In [12]:
def _same_dimension(u: Vector, v: Vector) -> None:
    if not isinstance(u, Vector) or not isinstance(v, Vector):
        raise TypeError("Both arguments must be Vectors")
    if len(u) != len(v):
        raise ValueError("Vector dimensions must match")


def project(u: Vector, onto: Vector) -> Vector:
    _same_dimension(u, onto)
    denom = onto * onto
    if denom == 0:
        raise ValueError("Cannot project onto the zero vector")
    return ((u * onto) / denom) * onto


def reject(u: Vector, onto: Vector) -> Vector:
    return u - project(u, onto)


def angle_between(u: Vector, v: Vector) -> float:
    _same_dimension(u, v)
    lengths = abs(u) * abs(v)
    if lengths == 0:
        raise ValueError("Angle is undefined for a zero vector")
    cosine = (u * v) / lengths
    return acos(max(-1.0, min(1.0, cosine)))

### Executable examples and verification

In [13]:
u = Vector(3, 4)
v = Vector(1, 0)
p, r = project(u, v), reject(u, v)
assert vector_close(p, Vector(3, 0))
assert vector_close(r, Vector(0, 4))
assert vector_close(p + r, u)
assert isclose(r * v, 0.0, abs_tol=1e-12)
assert isclose(angle_between(Vector(1, 0), Vector(0, 1)), pi/2)
assert isclose(angle_between(Vector(1, 0), Vector(-1, 0)), pi)
assert expect_raises(ValueError, lambda: project(u, Vector(0, 0)))
assert expect_raises(ValueError, lambda: angle_between(u, Vector(0, 0)))
assert expect_raises(ValueError, lambda: angle_between(u, Vector(1, 2, 3)))
print("projection:", p, "rejection:", r, "angle (radians):", angle_between(u, v))

projection: Vector(3.0, 0.0) rejection: Vector(0.0, 4.0) angle (radians): 0.9272952180016123


**Key takeaway / extension.** Defensive numerical programming means distinguishing an exact mathematical zero check from a domain-specific near-zero tolerance. Decide and document the tolerance according to input scale and application.

---
## Problem 07 — Normalization, distance, and interpolation contracts

**Challenge.** Implement named functions `unit(v)`, `distance(a,b)`, and `lerp(a,b,t)` using unary `abs`, subtraction, scalar multiplication, addition and division. Reject zero normalization, mismatched dimensions, and invalid interpolation parameters; support a documented optional extrapolation mode.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

A unit vector is `v / |v|`; distance is `|a−b|`. Linear interpolation uses `a + t(b−a)`: with `t=0` it returns `a`, with `t=1` it returns `b`, and otherwise it lies on their line. A strict interpolation API rejects `t` outside `[0,1]`, while `extrapolate=True` explicitly permits it. Reject `bool` parameters because Python otherwise treats `True` as `1`.

In [14]:
def unit(v: Vector) -> Vector:
    if not isinstance(v, Vector):
        raise TypeError("Expected a Vector")
    magnitude = abs(v)
    if magnitude == 0:
        raise ValueError("Cannot normalize the zero vector")
    return v / magnitude


def distance(a: Vector, b: Vector) -> float:
    _same_dimension(a, b)
    return abs(a - b)


def lerp(a: Vector, b: Vector, t: Real, *, extrapolate=False) -> Vector:
    _same_dimension(a, b)
    if isinstance(t, bool) or not isinstance(t, Real):
        raise TypeError("t must be a real non-bool number")
    if not extrapolate and not 0 <= t <= 1:
        raise ValueError("t must lie in [0,1]; request extrapolation explicitly")
    return a + t * (b - a)

### Executable examples and verification

In [15]:
a, b = Vector(0, 2), Vector(10, 12)
assert vector_close(unit(Vector(3, 4)), Vector(.6, .8))
assert isclose(abs(unit(Vector(3, 4))), 1.0)
assert distance(a, b) == hypot(10, 10)
assert lerp(a, b, 0) == a and lerp(a, b, 1) == b
assert lerp(a, b, .25) == Vector(2.5, 4.5)
assert lerp(a, b, 2, extrapolate=True) == Vector(20, 22)
assert expect_raises(ValueError, lambda: unit(Vector(0, 0)))
assert expect_raises(ValueError, lambda: lerp(a, b, 2))
assert expect_raises(TypeError, lambda: lerp(a, b, True))
print("normalized:", unit(Vector(3, 4)), "25% interpolation:", lerp(a, b, .25))

normalized: Vector(0.6, 0.8) 25% interpolation: Vector(2.5, 4.5)


**Key takeaway / extension.** `abs(v)` here represents Euclidean norm because `__abs__` explicitly defines it. This semantic choice should be part of a class's public contract, not a surprise hidden inside a magic method.

---
## Problem 08 — Enforce affine types: points are not vectors

**Challenge.** Design a `Point` wrapper that allows `Point + Vector -> Point`, `Vector + Point -> Point` (by reflected dispatch), `Point - Point -> Vector`, and `Point - Vector -> Point`, but forbids `Point + Point` and scaling a point. Validate dimensions and preserve the distinction between positions and displacements.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Points are locations; vectors are displacements. Their legal operators form an affine-space API. Implement operations on `Point` and let `Vector.__add__` return `NotImplemented` for a point so that `Point.__radd__` can handle `Vector + Point`. A point-plus-point sum is not intrinsically meaningful, so reject it. A frozen wrapper holding the immutable `Vector` avoids aliasing concerns.

In [16]:
@dataclass(frozen=True, slots=True)
class Point:
    coordinates: Vector

    def __post_init__(self):
        if not isinstance(self.coordinates, Vector):
            raise TypeError("Point coordinates must be a Vector")

    def __add__(self, other):
        if not isinstance(other, Vector) or len(other) != len(self.coordinates):
            return NotImplemented
        return Point(self.coordinates + other)

    def __radd__(self, other):
        if isinstance(other, Vector):
            return self + other
        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, Point) and len(other.coordinates) == len(self.coordinates):
            return self.coordinates - other.coordinates
        if isinstance(other, Vector) and len(other) == len(self.coordinates):
            return Point(self.coordinates - other)
        return NotImplemented

### Executable examples and verification

In [17]:
origin = Point(Vector(0, 0, 0))
move = Vector(2, -1, 3)
destination = Point(Vector(2, -1, 3))
assert origin + move == destination
assert move + origin == destination
assert destination - origin == move
assert destination - move == origin
assert expect_raises(TypeError, lambda: origin + destination)
assert expect_raises(TypeError, lambda: 2 * origin)
assert expect_raises(TypeError, lambda: origin + Vector(1, 2))
assert expect_raises(TypeError, lambda: Point("not a vector"))
print("destination:", destination, "displacement:", destination - origin)

destination: Point(coordinates=Vector(2, -1, 3)) displacement: Vector(2, -1, 3)


**Key takeaway / extension.** Domain modeling can prevent mathematically invalid expressions before they spread into production code. Avoid automatically treating two points as two vectors merely because their coordinates look alike.

---
## Problem 09 — Contrast immutable `+=` with atomic mutable `+=`

**Challenge.** Demonstrate that `x += y` on the immutable vector rebinds `x` without changing an existing alias. Build a mutable counterpart with `__iadd__` that preserves identity and updates aliases, but does not partially update on invalid inputs. Compare with a list and a tuple.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

`+=` first offers `__iadd__` and can fall back to `__add__`; the syntax **does not guarantee mutation**. Our frozen Vector has no `__iadd__`, so `x += y` rebinds the variable to the result of `x+y`. In contrast, a mutable class can validate and compute the whole replacement first, then commit it atomically and `return self`. Unsupported operands return `NotImplemented` so fallback can produce a `TypeError`; validation must precede mutation.

In [18]:
class MutableVector:
    def __init__(self, *components):
        self._components = list(Vector(*components).components)

    @property
    def components(self):
        return tuple(self._components)  # Protect internal list from external mutation.

    def __repr__(self):
        return f"MutableVector{self.components!r}"

    def __add__(self, other):
        if not isinstance(other, MutableVector) or len(other._components) != len(self._components):
            return NotImplemented
        return MutableVector(*(a + b for a, b in zip(self._components, other._components)))

    def __iadd__(self, other):
        if not isinstance(other, MutableVector) or len(other._components) != len(self._components):
            return NotImplemented
        replacement = [a + b for a, b in zip(self._components, other._components)]
        self._components = replacement
        return self

### Executable examples and verification

In [19]:
immutable = Vector(1, 2)
immutable_alias = immutable
immutable += Vector(10, 20)
assert immutable == Vector(11, 22)
assert immutable_alias == Vector(1, 2)
assert immutable is not immutable_alias

mutable = MutableVector(1, 2)
mutable_alias = mutable
mutable_identity = id(mutable)
mutable += MutableVector(10, 20)
assert id(mutable) == mutable_identity and mutable is mutable_alias
assert mutable_alias.components == (11, 22)
before = mutable.components
assert expect_raises(TypeError, lambda: mutable.__iadd__("wrong") + 0)
assert mutable.components == before
assert expect_raises(TypeError, lambda: mutable + MutableVector(1))

lst = [1, 2]
lst_alias = lst
lst += [3]
assert lst is lst_alias and lst_alias == [1, 2, 3]
tup = (1, 2)
tup_alias = tup
tup += (3,)
assert tup is not tup_alias and tup_alias == (1, 2)
print("immutable alias:", immutable_alias, "new binding:", immutable)
print("mutable alias sees update:", mutable_alias)

immutable alias: Vector(1, 2) new binding: Vector(11, 22)
mutable alias sees update: MutableVector(11, 22)


**Key takeaway / extension.** The direct `__iadd__` failure check above exercises a fallback path artificially; in real code, prefer testing the operator itself as well (done in the next cell). Never use `id` literals from a different process as expected answers—compare object identity in the current run.

In [20]:
# Additional operator-level failure test: Python handles both fallback attempts.
mutable_before = mutable.components
try:
    mutable += MutableVector(1)
except TypeError:
    pass
else:
    raise AssertionError("Dimension mismatch should fail")
assert mutable.components == mutable_before
print("Failed in-place addition left mutable data unchanged.")

Failed in-place addition left mutable data unchanged.


---
## Problem 10 — Use operator overloading for a `Family` aggregate

**Challenge.** Extend the source's nonnumeric `Family += child` example. Implement validated `Person` and `Family` types; make `family + child` return an independent family and `family += child` modify the original object. Reject wrong types and duplicate children. Test aliasing, copying, and rollback on invalid input.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

An operator should express a well-defined domain operation. Here `+` returns a fresh aggregate (no aliasing of the children's list), and `+=` mutates the existing one. `Person` is a frozen value object with a nonempty name; duplicated value objects are rejected as children by the chosen illustrative business rule. This rule is **an API example**, not a universal rule about families. The source example appends arbitrary objects without checking; the enhanced version adds validation.

In [21]:
@dataclass(frozen=True, slots=True)
class Person:
    name: str

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("Person requires a nonblank name")


class Family:
    def __init__(self, mother: Person, father: Person, children=()):
        if not isinstance(mother, Person) or not isinstance(father, Person):
            raise TypeError("Both parents must be Person instances")
        self.mother, self.father = mother, father
        self._children = []
        for child in children:
            self._validate_child(child)
            self._children.append(child)

    @property
    def children(self):
        return tuple(self._children)

    def _validate_child(self, child):
        if not isinstance(child, Person):
            raise TypeError("A child must be a Person")
        if child in self._children:
            raise ValueError("Duplicate child in this example's registry")

    def __add__(self, other):
        if not isinstance(other, Person):
            return NotImplemented
        self._validate_child(other)
        return Family(self.mother, self.father, (*self.children, other))

    def __iadd__(self, other):
        if not isinstance(other, Person):
            return NotImplemented
        self._validate_child(other)
        self._children.append(other)
        return self

### Executable examples and verification

In [22]:
original = Family(Person("Mara"), Person("Alex"))
original_alias = original
copy_with_child = original + Person("Sam")
assert original.children == ()
assert copy_with_child.children == (Person("Sam"),)
assert copy_with_child is not original
old_id = id(original)
original += Person("Riley")
assert original is original_alias and id(original) == old_id
assert original.children == (Person("Riley"),)
assert copy_with_child.children == (Person("Sam"),)
assert expect_raises(ValueError, lambda: original + Person("Riley"))
assert expect_raises(TypeError, lambda: original + "not a person")
assert expect_raises(ValueError, lambda: Person("  "))
assert original.children == (Person("Riley"),)
print("Original:", original.children, "independent copy:", copy_with_child.children)

Original: (Person(name='Riley'),) independent copy: (Person(name='Sam'),)


**Key takeaway / extension.** Use an immutable public `children` view and encapsulate mutable storage. Make the mutation-vs-copy difference explicit in docs and tests, especially when users hold references to the same aggregate.

---
## Problem 11 — Audit equality, hashing, and floating-point semantics

**Challenge.** Prove that immutable vectors are safe dictionary keys, inspect integer/float equality, and exhibit one floating-point computation for which exact component equality is inappropriate. Explain why approximate equality should be a separate helper rather than overriding `__eq__`.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

The frozen dataclass provides equality based on the tuple of coordinates and a compatible hash. An object used as a dictionary key must not change its hash while resident in the dictionary. Floats can represent `0.1+0.2` differently from `0.3`; approximate equality is often **not transitive**, so inserting tolerance inside `__eq__` can break equality/hash expectations. Use `vector_close` only for numerical comparisons, and keep key semantics exact.

In [23]:
def exact_key_example():
    lookup = {Vector(1, 2): "two-dimensional position"}
    return lookup[Vector(1.0, 2.0)], lookup


def approximate_vector_equal(a: Vector, b: Vector, *, atol=1e-12, rtol=1e-12):
    _same_dimension(a, b)
    return vector_close(a, b, rel_tol=rtol, abs_tol=atol)

### Executable examples and verification

In [24]:
value, mapping = exact_key_example()
assert value == "two-dimensional position"
assert len(mapping) == 1
assert Vector(1, 2) == Vector(1.0, 2.0)
assert hash(Vector(1, 2)) == hash(Vector(1.0, 2.0))
x, y = Vector(0.1 + 0.2, 2.0), Vector(0.3, 2.0)
assert x != y
assert approximate_vector_equal(x, y)
assert expect_raises(ValueError, lambda: approximate_vector_equal(Vector(1), Vector(1, 2)))
print("Exact equality:", x == y, "tolerance comparison:", approximate_vector_equal(x, y))

Exact equality: False tolerance comparison: True


**Key takeaway / extension.** Avoid approximate `__hash__` hacks. For spatial bucketing, quantize coordinates into an explicitly designed key type with clear boundary behavior rather than silently changing vector equality.

---
## Problem 12 — Build a safe vector aggregation API

**Challenge.** Implement `sum_vectors(iterable, dimension=...)` that works with generators, validates dimensions, and defines the empty-input result only if dimension is supplied. Show why built-in `sum([Vector(...), ...])` fails with its default numeric zero, and how an explicit `start` resolves it.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Python's `sum` starts at integer `0` by default; this Vector deliberately does **not** pretend `0 + v` is a valid mixed-type vector operation. Pass `start=Vector(0, ..., 0)` or define an explicit aggregation function. To handle one-shot generators, iterate once and determine dimension from the first item when no dimension is provided. Reject nonpositive dimension and mismatches instead of silently truncating.

In [25]:
def sum_vectors(items: Iterable[Vector], *, dimension: int | None = None) -> Vector:
    if dimension is not None and (isinstance(dimension, bool)
                                  or not isinstance(dimension, int)
                                  or dimension < 1):
        raise ValueError("dimension must be a positive integer")
    iterator = iter(items)
    if dimension is None:
        first = next(iterator, None)
        if first is None:
            raise ValueError("Cannot infer dimension from an empty iterable")
        if not isinstance(first, Vector):
            raise TypeError("Every item must be a Vector")
        total = Vector(*(0 for _ in first))
        iterator = chain((first,), iterator)  # Reinsert first without exhausting the generator.
    else:
        total = Vector(*(0 for _ in range(dimension)))
    for item in iterator:
        if not isinstance(item, Vector):
            raise TypeError("Every item must be a Vector")
        if len(item) != len(total):
            raise ValueError("Cannot aggregate different dimensions")
        total = total + item
    return total

### Executable examples and verification

In [26]:
items = [Vector(1, 2), Vector(3, 4), Vector(-1, 0)]
assert sum(items, start=Vector(0, 0)) == Vector(3, 6)
assert sum_vectors(iter(items)) == Vector(3, 6)
assert sum_vectors((Vector(i, 2 * i) for i in range(5))) == Vector(10, 20)
assert sum_vectors([], dimension=3) == Vector(0, 0, 0)
assert expect_raises(TypeError, lambda: sum(items))
assert expect_raises(ValueError, lambda: sum_vectors([]))
assert expect_raises(ValueError, lambda: sum_vectors([Vector(1, 2), Vector(3)]))
assert expect_raises(TypeError, lambda: sum_vectors([Vector(1), "bad"]))
print("Aggregated:", sum_vectors(items), "empty with dimension:", sum_vectors([], dimension=3))

Aggregated: Vector(3, 6) empty with dimension: Vector(0, 0, 0)


**Key takeaway / extension.** The implementation uses `itertools.chain` to preserve streaming behavior. A slightly streamlined variant with explicit O(d) additional storage and further tests follows immediately.

In [27]:
from itertools import chain


def sum_vectors_streaming(items: Iterable[Vector], *, dimension: int | None = None) -> Vector:
    """Single pass, O(d) extra storage, even for large generators."""
    if dimension is not None and (type(dimension) is not int or dimension < 1):
        raise ValueError("dimension must be a positive integer")
    iterator = iter(items)
    if dimension is None:
        first = next(iterator, None)
        if first is None:
            raise ValueError("Cannot infer an empty iterable's dimension")
        if not isinstance(first, Vector):
            raise TypeError("All items must be Vectors")
        dimension = len(first)
        iterator = chain((first,), iterator)
    result = Vector(*(0 for _ in range(dimension)))
    for item in iterator:
        if not isinstance(item, Vector):
            raise TypeError("All items must be Vectors")
        if len(item) != dimension:
            raise ValueError("Dimension mismatch")
        result = result + item
    return result

assert sum_vectors_streaming((Vector(i, -i) for i in range(1000))) == Vector(499500, -499500)
assert sum_vectors_streaming([], dimension=2) == Vector(0, 0)
print("Streaming aggregator passed generator and empty-input tests.")

Streaming aggregator passed generator and empty-input tests.


---
## Problem 13 — Simulate motion with overloaded operators

**Challenge.** Create a frozen `Particle(position, velocity, mass)` and a semi-implicit Euler step. Use `+`, `-`, `*`, `abs`, and scalar division for the update, momentum, and kinetic energy. Validate dimensions, mass, time step, and acceleration. Verify a small constant-gravity example and zero-acceleration motion.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Semi-implicit Euler updates velocity **first**: `v_new = v + a·dt`; then `p_new = p + v_new·dt`. This is intentionally not an exact solution for accelerated motion. Momentum is `m·v` and kinetic energy is `0.5·m·|v|²`. For numerical simulation, distinguish physical dimensions (units) from vector coordinate count; this example validates only coordinate count and scalar type, not physical units. Do not modify the input particle.

In [28]:
@dataclass(frozen=True, slots=True)
class Particle:
    position: Vector
    velocity: Vector
    mass: Real

    def __post_init__(self):
        if not isinstance(self.position, Vector) or not isinstance(self.velocity, Vector):
            raise TypeError("position and velocity must be Vectors")
        if len(self.position) != len(self.velocity):
            raise ValueError("position and velocity dimensions must match")
        if isinstance(self.mass, bool) or not isinstance(self.mass, Real) or self.mass <= 0:
            raise ValueError("mass must be a strictly positive real number")

    @property
    def momentum(self):
        return self.mass * self.velocity

    @property
    def kinetic_energy(self):
        return 0.5 * self.mass * abs(self.velocity)**2


def advance(particle: Particle, acceleration: Vector, dt: Real) -> Particle:
    if not isinstance(particle, Particle) or not isinstance(acceleration, Vector):
        raise TypeError("Expected Particle and Vector")
    if len(acceleration) != len(particle.position):
        raise ValueError("Acceleration dimension mismatch")
    if isinstance(dt, bool) or not isinstance(dt, Real):
        raise TypeError("dt must be a real non-bool number")
    if dt < 0:
        raise ValueError("dt cannot be negative")
    new_velocity = particle.velocity + acceleration * dt
    new_position = particle.position + new_velocity * dt
    return Particle(new_position, new_velocity, particle.mass)

### Executable examples and verification

In [29]:
initial = Particle(Vector(0, 10), Vector(2, 0), 3)
a = Vector(0, -10)
next_state = advance(initial, a, 0.5)
assert next_state.velocity == Vector(2.0, -5.0)
assert next_state.position == Vector(1.0, 7.5)
assert initial.position == Vector(0, 10)
assert next_state.momentum == Vector(6.0, -15.0)
assert isclose(initial.kinetic_energy, 6.0)
coast = advance(initial, Vector(0, 0), 2)
assert coast.position == Vector(4, 10)
assert expect_raises(ValueError, lambda: advance(initial, a, -1))
assert expect_raises(ValueError, lambda: advance(initial, Vector(1), 1))
assert expect_raises(ValueError, lambda: Particle(Vector(0), Vector(0), 0))
print("Semi-implicit step:", next_state, "kinetic energy:", next_state.kinetic_energy)

Semi-implicit step: Particle(position=Vector(1.0, 7.5), velocity=Vector(2.0, -5.0), mass=3) kinetic energy: 43.49999999999999


**Key takeaway / extension.** Numerical integration is sensitive to step size. This exercise checks API correctness and one update; it does not establish accuracy or long-term energy conservation of the simulation.

---
## Problem 14 — Design a polynomial arithmetic protocol

**Challenge.** Create an immutable dense `Polynomial` with coefficients in ascending power order. Implement addition, multiplication, reflected scalar multiplication and addition, evaluation using Horner's method, exponentiation by nonnegative integer, normalization of trailing zeros, and failure-path validation. Compare with `Vector`'s meaning of `*`.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Operator symbols may have different mathematically coherent meanings by type. Polynomial multiplication is **convolution**, not coordinatewise vector scaling or dot product: the coefficient of `x^k` is `Σ aᵢbₖ₋ᵢ`. Trim trailing zeros so equality is canonical; preserve a single `(0,)` for the zero polynomial. Evaluate via Horner, `(...((aₙ x+aₙ₋₁)x+...)x+a₀)`, with O(n) multiplications. Implement exponentiation by squaring in O(log exponent) polynomial multiplications. Scalar operations are supported only for real nonbool operands.

In [30]:
@dataclass(frozen=True, slots=True, init=False)
class Polynomial:
    coefficients: tuple[Real, ...]

    def __init__(self, *coefficients: Real):
        if not coefficients:
            raise ValueError("Provide at least one coefficient")
        if any(isinstance(c, bool) or not isinstance(c, Real) for c in coefficients):
            raise TypeError("Polynomial coefficients must be real non-bool values")
        values = list(coefficients)
        while len(values) > 1 and values[-1] == 0:
            values.pop()
        object.__setattr__(self, "coefficients", tuple(values))

    @property
    def degree(self):
        return len(self.coefficients) - 1

    def __add__(self, other):
        if isinstance(other, Real) and not isinstance(other, bool):
            other = Polynomial(other)
        if not isinstance(other, Polynomial):
            return NotImplemented
        size = max(len(self.coefficients), len(other.coefficients))
        return Polynomial(*(
            (self.coefficients[i] if i < len(self.coefficients) else 0) +
            (other.coefficients[i] if i < len(other.coefficients) else 0)
            for i in range(size)))

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        if isinstance(other, Real) and not isinstance(other, bool):
            return Polynomial(*(c * other for c in self.coefficients))
        if not isinstance(other, Polynomial):
            return NotImplemented
        result = [0] * (len(self.coefficients) + len(other.coefficients) - 1)
        for i, a in enumerate(self.coefficients):
            for j, b in enumerate(other.coefficients):
                result[i+j] += a * b
        return Polynomial(*result)

    def __rmul__(self, other):
        return self * other

    def __call__(self, x):
        if isinstance(x, bool) or not isinstance(x, Real):
            raise TypeError("Evaluation point must be a real non-bool value")
        value = 0
        for coeff in reversed(self.coefficients):
            value = value * x + coeff
        return value

    def __pow__(self, exponent):
        if type(exponent) is not int or exponent < 0:
            return NotImplemented
        result, base = Polynomial(1), self
        while exponent:
            if exponent & 1:
                result = result * base
            base = base * base
            exponent >>= 1
        return result

### Executable examples and verification

In [31]:
p = Polynomial(1, 2, 0, 0)  # 1 + 2x
q = Polynomial(3, 0, 1)     # 3 + x^2
assert p == Polynomial(1, 2) and p.degree == 1
assert p + q == Polynomial(4, 2, 1)
assert p + 5 == 5 + p == Polynomial(6, 2)
assert p * q == Polynomial(3, 6, 1, 2)
assert 2 * p == p * 2 == Polynomial(2, 4)
assert p(3) == 7 and q(2) == 7
assert (Polynomial(1, 1) ** 3) == Polynomial(1, 3, 3, 1)
assert p ** 0 == Polynomial(1)
assert Polynomial(0, 0, 0) == Polynomial(0)
assert expect_raises(ValueError, lambda: Polynomial())
assert expect_raises(TypeError, lambda: p + [1, 2])
assert expect_raises(TypeError, lambda: p ** -1)
assert expect_raises(TypeError, lambda: Polynomial(True))
print("Polynomial product coefficients:", (p * q).coefficients, "p(3):", p(3))

Polynomial product coefficients: (3, 6, 1, 2) p(3): 7


**Key takeaway / extension.** The implementation uses naive O(mn) convolution, ideal for teaching. Specialized polynomial packages may use faster algorithms for large degrees. Polynomial equality is exact; floating coefficients may still need a separate approximate comparison.

---
## Problem 15 — Implement conventional matrix `@`

**Challenge.** Define a rectangular immutable `Matrix` supporting `Matrix @ Vector` and `Matrix @ Matrix`. Require valid shapes and meaningful errors, implement transpose and identity, and check associativity `(A@B)@v = A@(B@v)` for compatible shapes. Contrast this with the custom `Vector @ Vector` cross product.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

An m×n matrix times an n-vector yields an m-vector by dotting each row with the vector. An m×n matrix times an n×p matrix yields an m×p matrix; each entry is a row-column dot product. Use `NotImplemented` for unsupported binary types or shape mismatches (the ultimate operator error is `TypeError`, as in the lesson). Validate rectangular rows at construction so no operation encounters ragged input. Keep rows immutable. The `Matrix @` API now matches the conventional interpretation of `@`.

In [32]:
@dataclass(frozen=True, slots=True, init=False)
class Matrix:
    rows: tuple[tuple[Real, ...], ...]

    def __init__(self, rows):
        materialized = tuple(tuple(row) for row in rows)
        if not materialized or not materialized[0]:
            raise ValueError("Matrix must have at least one row and column")
        width = len(materialized[0])
        if any(len(row) != width for row in materialized):
            raise ValueError("Rows must all have the same length")
        if any(isinstance(x, bool) or not isinstance(x, Real)
               for row in materialized for x in row):
            raise TypeError("Matrix entries must be real non-bool values")
        object.__setattr__(self, "rows", materialized)

    @property
    def shape(self):
        return len(self.rows), len(self.rows[0])

    @property
    def T(self):
        return Matrix(zip(*self.rows))

    @classmethod
    def identity(cls, n):
        if type(n) is not int or n <= 0:
            raise ValueError("Identity size must be a positive integer")
        return cls(tuple(tuple(int(i == j) for j in range(n)) for i in range(n)))

    def __matmul__(self, other):
        if isinstance(other, Vector):
            if self.shape[1] != len(other):
                return NotImplemented
            return Vector(*(sum(a*b for a, b in zip(row, other)) for row in self.rows))
        if isinstance(other, Matrix):
            if self.shape[1] != other.shape[0]:
                return NotImplemented
            columns = tuple(zip(*other.rows))
            return Matrix(tuple(tuple(sum(a*b for a, b in zip(row, col))
                                      for col in columns) for row in self.rows))
        return NotImplemented

### Executable examples and verification

In [33]:
A = Matrix(((1, 2, 3), (4, 5, 6)))   # 2 x 3
B = Matrix(((1, 0), (0, 1), (1, 1)))  # 3 x 2
v = Vector(2, -1)
assert A.shape == (2, 3) and A.T.shape == (3, 2)
assert A @ Vector(1, 0, -1) == Vector(-2, -2)
assert A @ B == Matrix(((4, 5), (10, 11)))
assert (A @ B) @ v == A @ (B @ v)
assert Matrix.identity(2) @ v == v
assert expect_raises(ValueError, lambda: Matrix(((1, 2), (3,))))
assert expect_raises(TypeError, lambda: A @ v)
assert expect_raises(TypeError, lambda: A @ "not a matrix")
assert expect_raises(ValueError, lambda: Matrix.identity(0))
print("A @ B:", (A @ B).rows, "A @ (B @ v):", A @ (B @ v))

A @ B: ((4, 5), (10, 11)) A @ (B @ v): Vector(3, 9)


**Key takeaway / extension.** A cross product and matrix multiplication have different domains and laws even though both can use `@`. Write explicit tests and documentation when an educational operator convention differs from a mainstream library convention.

---
## Problem 16 — Detect numerical overflow and cancellation

**Challenge.** Contrast a naive Euclidean norm with `hypot`, and ordinary floating-point summation with `fsum`. Explain why naive squared magnitudes can overflow even when the true norm fits a float. Show why a cancellation example exposes a limitation in our exact/integer-friendly `Vector.__mul__` dot implementation.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

`sqrt(sum(x*x))` squares input magnitudes before taking a root, which can overflow to infinity. `hypot` rescales internally for many such cases. Floating summation also loses small terms in the presence of huge terms: a naive left-to-right floating-point accumulator loses the middle `1` in `(1e16, 1, -1e16)`, whereas `fsum` recovers it. Some recent Python versions also use compensated summation for built-in `sum`, so its result is interpreter-dependent. The core `Vector.__mul__` deliberately uses Python `sum` to preserve normal exact integer behavior; an explicitly floating numerical API may choose `fsum` or a specialized BLAS routine instead. Neither approach solves every overflow or conditioning problem.

In [34]:
def naive_norm(v: Vector):
    return sqrt(sum(x * x for x in v))


def accurate_float_dot(a: Vector, b: Vector) -> float:
    _same_dimension(a, b)
    return fsum(float(x) * float(y) for x, y in zip(a, b))

### Executable examples and verification

In [35]:
huge = Vector(1e200, 1e200)
assert naive_norm(huge) == float("inf")
assert isclose(abs(huge) / 1e200, sqrt(2), rel_tol=1e-15)
a = Vector(1e16, 1.0, -1e16)
b = Vector(1.0, 1.0, 1.0)
regular_dot = a * b  # Python's sum may already compensate on recent interpreters.
naive_dot = 0.0
for left, right in zip(a, b):
    naive_dot += left * right
precise_dot = accurate_float_dot(a, b)
assert naive_dot == 0.0
assert precise_dot == 1.0
assert isclose(regular_dot, precise_dot, abs_tol=1.0)
large_integers = Vector(10**30, 2) * Vector(10**30, 3)
assert large_integers == 10**60 + 6
print("Naive norm:", naive_norm(huge), "stable norm:", abs(huge))
print("Naive loop dot:", naive_dot, "built-in sum dot:", regular_dot, "fsum dot:", precise_dot)

Naive norm: inf stable norm: 1.414213562373095e+200
Naive loop dot: 0.0 built-in sum dot: 1.0 fsum dot: 1.0


**Key takeaway / extension.** For extreme real-number arithmetic, test overflow, underflow, nonfinite values, and cancellation explicitly. Higher precision and better algorithms are separate design decisions, not free properties of operator overloading.

---
## Problem 17 — Verify algebraic laws with deterministic randomized tests

**Challenge.** Use a fixed random seed to test additive associativity, subtraction inverses, scalar distributivity, dot-product symmetry/linearity, and 3D cross-product orthogonality on many random small integer vectors. Include a counterexample to an invalid alleged law and explain why property tests must not assert exact associativity on arbitrary floats.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Property-based thinking checks families of inputs rather than a few handpicked examples. Integer vectors keep the arithmetic exact and avoid nonassociative floating behavior. Check the correct operator contracts; for example the cross product is **anti**commutative, not commutative. `Random(seed)` makes failures reproducible without installing Hypothesis. For floating tests, use error tolerances tied to scale and conditioning rather than `==`.

In [36]:
def randomized_operator_audit(*, trials=300, seed=2026):
    rng = Random(seed)
    for _ in range(trials):
        a, b, c = (Vector(*(rng.randint(-6, 6) for _ in range(3)))
                   for _ in range(3))
        scalar = rng.randint(-4, 4)
        assert (a + b) + c == a + (b + c)
        assert (a - b) + b == a
        assert scalar * (a + b) == scalar * a + scalar * b
        assert a * b == b * a
        assert (a + b) * c == a * c + b * c
        assert a @ b == -(b @ a)
        assert (a @ b) * a == 0
        assert (a @ b) * b == 0
    return trials

### Executable examples and verification

In [37]:
assert randomized_operator_audit() == 300
x, y = Vector(1, 0, 0), Vector(0, 1, 0)
assert x @ y != y @ x  # Cross product is NOT commutative.
assert (1e16 + -1e16) + 1.0 != 1e16 + (-1e16 + 1.0)
print("Passed 300 seeded integer-vector trials and the invalid-law counterexamples.")

Passed 300 seeded integer-vector trials and the invalid-law counterexamples.


**Key takeaway / extension.** Property checks complement, not replace, targeted edge-case tests. For production-scale testing, consider Hypothesis as an optional dependency and use its shrinking/reproduction facilities.

---
## Problem 18 — Capstone: modified Gram–Schmidt orthonormalization

**Challenge.** Given a nonempty collection of linearly independent vectors of one dimension, return orthonormal basis vectors using subtraction, projection, dot product, division, and unary `abs`. Detect dependent inputs, choose a numerical tolerance, and verify unit norms, pairwise orthogonality, and reconstruction of a sample vector. Explain why this is an algorithmic stress test for the overloaded arithmetic API.

**Try first:** predict the behavior and sketch a solution before revealing/running the cells below.

### Worked solution and reasoning

Modified Gram–Schmidt processes each candidate `v`, removing the component along each **already normalized** basis vector, `residual ← residual − (residual·q)q`, then normalizes the residual. If its norm is near zero, the candidate is dependent or numerically unresolved at the selected tolerance, so raise `ValueError`. Require finite, nonzero tolerance. This is an educational implementation; robust numerical linear algebra on ill-conditioned matrices generally uses Householder QR/SVD through established libraries.

In [38]:
def orthonormalize(vectors: Iterable[Vector], *, tol: float=1e-12) -> tuple[Vector, ...]:
    if not isinstance(tol, Real) or isinstance(tol, bool) or not 0 < tol < float("inf"):
        raise ValueError("tol must be a finite, positive real number")
    iterator = iter(vectors)
    first = next(iterator, None)
    if first is None:
        raise ValueError("Need at least one vector")
    if not isinstance(first, Vector):
        raise TypeError("Inputs must be Vectors")
    dimension = len(first)
    basis = []
    for candidate in chain((first,), iterator):
        if not isinstance(candidate, Vector):
            raise TypeError("Inputs must be Vectors")
        if len(candidate) != dimension:
            raise ValueError("All input dimensions must agree")
        residual = candidate
        for q in basis:
            residual = residual - (residual * q) * q
        norm = abs(residual)
        if norm <= tol:
            raise ValueError("Linearly dependent or numerically unresolved input")
        basis.append(residual / norm)
    return tuple(basis)


def reconstruct(coefficients: Iterable[Real], basis: tuple[Vector, ...]) -> Vector:
    coeffs = tuple(coefficients)
    if len(coeffs) != len(basis) or not basis:
        raise ValueError("Coefficient/basis size mismatch or empty basis")
    return sum_vectors_streaming((c * q for c, q in zip(coeffs, basis)),
                                 dimension=len(basis[0]))

### Executable examples and verification

In [39]:
inputs = (Vector(1, 1, 0), Vector(1, 0, 1), Vector(0, 1, 1))
Q = orthonormalize(inputs)
assert len(Q) == 3
for i, q in enumerate(Q):
    assert isclose(abs(q), 1, abs_tol=1e-12)
    for j, r in enumerate(Q):
        assert isclose(q * r, 1.0 if i == j else 0.0, abs_tol=1e-12)
target = Vector(3, -2, 5)
coefficients = tuple(target * q for q in Q)
assert vector_close(reconstruct(coefficients, Q), target)
assert expect_raises(ValueError, lambda: orthonormalize([Vector(1, 0), Vector(2, 0)]))
assert expect_raises(ValueError, lambda: orthonormalize([]))
assert expect_raises(ValueError, lambda: orthonormalize([Vector(1), Vector(1, 0)]))
assert expect_raises(ValueError, lambda: orthonormalize([Vector(1)], tol=0))
print("Orthonormal basis:", Q)
print("Reconstruction:", reconstruct(coefficients, Q), "original:", target)

Orthonormal basis: (Vector(0.7071067811865475, 0.7071067811865475, 0.0), Vector(0.40824829046386313, -0.40824829046386296, 0.8164965809277261), Vector(-0.5773502691896257, 0.5773502691896258, 0.5773502691896256))
Reconstruction: Vector(3.0000000000000013, -2.0000000000000004, 5.000000000000002) original: Vector(3, -2, 5)


**Key takeaway / extension.** Time complexity for n vectors of dimension d is approximately O(n²d), not counting Python object construction. This capstone also demonstrates why a clean, consistently validated arithmetic protocol pays off in larger algorithms.

---
## Bonus worked examples and interview-style traps

The problems above form the main progression. These extra cells show subtle dispatch rules, error contracts, and performance interpretation. They are intentionally executable, rather than merely theoretical.

In [40]:
# Extra A: NotImplemented vs NotImplementedError.
class WrongFallback:
    def __add__(self, other):
        raise NotImplementedError("This stops dispatch immediately")

class AcceptFallback:
    def __radd__(self, other):
        return "accepted"

assert expect_raises(NotImplementedError, lambda: WrongFallback() + AcceptFallback())
assert LeftOperand() + RightOperand() == "reflected success"
print("A raised NotImplementedError cannot trigger normal reflected fallback.")

A raised NotImplementedError cannot trigger normal reflected fallback.


In [41]:
# Extra B: the classic tuple-in-list augmented-assignment trap.
container = ([1, 2],)
try:
    container[0] += [3]
except TypeError:
    pass
else:
    raise AssertionError("Tuple item assignment should fail")
assert container == ([1, 2, 3],)
print("The inner list mutated before tuple reassignment failed:", container)

The inner list mutated before tuple reassignment failed: ([1, 2, 3],)


In [42]:
# Extra C: useful special-method invocation patterns.
v = Vector(2, -5, 1)
assert (v.__mul__("wrong") is NotImplemented)
assert (v.__add__(Vector(1)) is NotImplemented)
assert (v.__matmul__(Vector(1, 0)) is NotImplemented)
assert vector_close(v / 2, Vector(1, -2.5, .5))
assert -(-v) == v
assert abs(-v) == abs(v)
print("Unsupported direct dunder calls returned NotImplemented; operators raise TypeError.")

Unsupported direct dunder calls returned NotImplemented; operators raise TypeError.


In [43]:
# Extra D: dot/cross combination: vector triple-product identity in R^3.
a, b, c = Vector(1, 2, 3), Vector(4, -1, 2), Vector(2, 0, 1)
left = a @ (b @ c)
right = b * (a * c) - c * (a * b)
assert left == right  # a x (b x c) = b(a.c) - c(a.b)
assert isclose(a * (b @ c), (a @ b) * c)
print("Vector triple product:", left, "scalar triple product:", a * (b @ c))

Vector triple product: Vector(4, -5, 2) scalar triple product: 5


### Optional microbenchmark (observational, not a pass/fail performance test)

Time measurements depend on hardware and interpreter and therefore should never be hardcoded as a correctness assertion. Here we compare a list-based Python baseline with the immutable `Vector` operator, observing allocation/dispatch overhead on moderate inputs.

In [44]:
from operator import add

size = 2000
lhs, rhs = Vector(*range(size)), Vector(*range(size, 2 * size))
vector_time = timeit(lambda: lhs + rhs, number=100)
list_time = timeit(lambda: list(map(add, lhs.components, rhs.components)), number=100)
assert (lhs + rhs)[-1] == lhs[-1] + rhs[-1]
print(f"100 runs, dimension={size}: Vector addition {vector_time:.4f}s; list baseline {list_time:.4f}s")
print("Interpret timings as environment-specific observations, not a universal ranking.")

100 runs, dimension=2000: Vector addition 0.1170s; list baseline 0.0205s
Interpret timings as environment-specific observations, not a universal ranking.


---
## Final self-check: mastery checklist

- [ ] I can predict normal/reflected/subclass operator dispatch and use `NotImplemented` correctly.
- [ ] I can define an immutable, well-validated vector and distinguish operator failure from constructor failure.
- [ ] I can explain why `*` has two meanings in this notebook and why `@` is nonstandard for the vector cross product.
- [ ] I can prove algebraic laws and write edge, identity, and aliasing tests.
- [ ] I can choose deliberately whether augmented assignment mutates or rebinds.
- [ ] I can build a nonnumeric arithmetic protocol with explicit domain validation.
- [ ] I can recognize floating error, overflow, and limitations of naive algorithms.
- [ ] I can compose overloaded operations into geometry, physics, matrix and orthonormalization algorithms.

**Further challenges (deliberately unsolved):** implement an exact rational-coordinate vector using `fractions.Fraction`; implement an explicit `dot`/`cross` API without overloaded multiplication; add `Matrix.__add__` and inverse for a nonsingular 2×2 matrix; compare modified Gram–Schmidt with reorthogonalization on nearly dependent data; write Hypothesis strategies for valid/mismatched dimensions.

**Source alignment:** Expanded from the provided lesson on Python arithmetic operators and its `Vector` and `Family` examples. The lesson illustrates a placeholder `__matmul__` for a cross product; this notebook supplies a complete three-dimensional definition and identifies the nonstandard convention. Its mutable `__iadd__` example is presented separately from the immutable design used for the main algorithms. This notebook is an expanded original set of exercises and solutions, not a reproduction of the lesson.